In [1]:
from catboost import CatBoostClassifier, Pool
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import re
import math
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GroupKFold
from sentence_transformers import SentenceTransformer
from scipy.stats import mode
import joblib
from features import prepare_data, select_numeric, select_char_text, select_full_text, BadRequestError

pd.set_option('display.max_columns', None)


In [3]:
df = pd.read_csv('df0.csv')

In [4]:
numeric_features_ordered = [
        'items_total_price', 'items_mean_price', 'items_price_std',
        'items_min_price', 'items_max_price', 'items_price_range',
        'items_vs_amount', 'amount_log', 'terminal_name_len',
        'terminal_desc_len', 'items_text_len', 'amount_per_item',
        'items_price_skew'
    ]
text_col = ['char_text']

In [5]:
X_cat_final = df[numeric_features_ordered + ['full_text']]
text_col = ['char_text']
X_svc_final = df[numeric_features_ordered + ['char_text']]
X_lr_char = df[numeric_features_ordered + ['full_text']]



In [12]:
def select_char_text(X):
    return X['char_text']

def select_numeric(X):
    return X[numeric_features_ordered]

def select_full_text(X):
    return X['full_text']

In [6]:
text_transformer = Pipeline([
    ('select', FunctionTransformer(select_char_text, validate=False)),
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2,5),
        max_features=7000,
        min_df=1,
        max_df=0.1,
        sublinear_tf=True
    ))
])


features = ColumnTransformer(
    transformers=[
        ('char', text_transformer, text_col),
        ('num', StandardScaler(), numeric_features_ordered)
    ]
)

svc = LinearSVC(C=1.5, max_iter=4000)

calibrated_svc = CalibratedClassifierCV(
    estimator=svc,
    cv=3,
    method='sigmoid'
)

model_svc = Pipeline([
    ('features', features),
    ('clf', calibrated_svc)
])

In [7]:
text_union = FeatureUnion([
    ('term_name', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,3), min_df=1, max_df=0.1)),
    ('term_desc', TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=1, max_df=0.1)),
    ('items', TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=1, max_df=0.1))
])

full_pipeline = FeatureUnion([
    ('text', Pipeline([
        ('selector', FunctionTransformer(select_full_text, validate=False)),
        ('tfidf', text_union)
    ])),
    ('numeric', Pipeline([
        ('selector', FunctionTransformer(select_numeric, validate=False)),
        ('scaler', StandardScaler())
    ]))
])

model_lr_char = Pipeline([
    ('features', full_pipeline),
    ('clf', LogisticRegression(
        C=2.5,
        multi_class='multinomial',
        max_iter=2000,
        solver='saga'
    ))
])


In [8]:
model_cat = CatBoostClassifier(
        iterations=400,
        depth=6,
        learning_rate=0.1,
        loss_function='MultiClass',
        eval_metric='Accuracy',
        random_seed=42,
        verbose=False
    )

In [9]:
le = LabelEncoder()
y_encoded = le.fit_transform(df['true_mcc'])

In [10]:
model_svc.fit(X_svc_final, y_encoded)
model_lr_char.fit(df, y_encoded)
model_cat.fit(X_cat_final, y_encoded, text_features=[X_cat_final.columns.get_loc('full_text')])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [11]:
model_cat.save_model("/Users/mila/emmi-ffjrr_gmail-com/solution/model/model_cat.cbm")

In [12]:
joblib.dump(model_lr_char, "/Users/mila/emmi-ffjrr_gmail-com/solution/model/model_lr_char.pkl")

['/Users/mila/emmi-ffjrr_gmail-com/solution/model/model_lr_char.pkl']

In [13]:
joblib.dump(model_svc, "/Users/mila/emmi-ffjrr_gmail-com/solution/model/model_svc.pkl")

['/Users/mila/emmi-ffjrr_gmail-com/solution/model/model_svc.pkl']

In [14]:
joblib.dump(le, "/Users/mila/emmi-ffjrr_gmail-com/solution/model/le_mcc.pkl")

['/Users/mila/emmi-ffjrr_gmail-com/solution/model/le_mcc.pkl']